### Imports

In [1]:
import pandas as pd

In [2]:
from scripts.utils import write_fasta

In [3]:
from training_pLM.utils import calculate_metrics

/home/shaburova/mambaforge/envs/DBP-Finder/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/shaburova/mambaforge/envs/DBP-Finder/lib/python3.11/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/shaburova/mambaforge/envs/DBP-Finder/lib/python3.11/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


### Model weights

In [5]:
from huggingface_hub import snapshot_download

In [6]:
model_name = "camelStyle/DBP-Finder" # Replace with the model you want to download
local_dir = "weights"

snapshot_download(repo_id=model_name, local_dir=local_dir);

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

model.pth:   0%|          | 0.00/84.3M [00:00<?, ?B/s]

'/storage/shaburova/DBP-Finder/weights'

### Example dataset inference

In [7]:
df = pd.read_csv("data/embeddings/input_csv/pdb2272.csv")
df

,identifier,sequence,label
0,P13123,VKVKFKYKGEEKEVDTSKIKKVWRVGKMVSFTYDDNGKTGRGAVSE...,1
1,P03211,MSDEGPGTGPGNGLGEKGDTSGPEGSGGSGPQRRGGDNHGRGRGRG...,1
2,P03023,MKPVTLYDVAEYAGVSYQTVSRVVNQASHVSAKTREKVEAAMAELN...,1
3,Q38087,MKEFYLTVEQIGDSIFERYIDSNGRERTREVEYKPSLFAHCPESQA...,1
4,P22670,MATQAYTELQAAPPPSQPPQAPPQAQPQPPPPPPPAAPQPPQPPTA...,1
...,...,...,...
2267,P37471,MNFSRERTITEIQNDYKEQVERQNQLKKRRRKGLYRRLTVFGALVF...,0
2268,P07078,MVVVDKEIKKGQYYLVNGNVVRVTYVNGFDVYYLILKLHKRMICDR...,0
2269,P16793,MNPSTHVSSNGPTTPPHGPHTTFLPPTSPAPSTSSVAAATLCSPQR...,0
2270,P80484,MVRSGKKAVVLAAVAFCATSVVQKSHGFVPSPLRQRAAAAGAAAAS...,0


In [8]:
write_fasta(df, "data/fasta/pdb2272.fasta")

In [9]:
import subprocess

subprocess.run(
    f"python3 -m training_pLM.inference data/fasta/pdb2272.fasta dbp_finder_pdb2272_prediction --gpu 0",
    shell=True,
    check=True,  # Raises an error if the command fails
)

/home/shaburova/mambaforge/envs/DBP-Finder/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/shaburova/mambaforge/envs/DBP-Finder/lib/python3.11/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/shaburova/mambaforge/envs/DBP-Finder/lib/python3.11/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2025-12-29 11:55:42,696 - INFO - Starting pipeline...
2025-12-29 11:55:42,792 - INFO - Converting FASTA to dataframe...
2025-12-29 11:55:42,828 - WARNI

Predictions saved to data/prediction/dbp_finder_pdb2272_prediction.csv


CompletedProcess(args='python3 -m training_pLM.inference data/fasta/pdb2272.fasta dbp_finder_pdb2272_prediction --gpu 0', returncode=0)

In [10]:
df_pred = pd.read_csv("data/prediction/dbp_finder_pdb2272_prediction.csv")
df_pred = pd.merge(df, df_pred, on="identifier", how="inner")
df_pred

,identifier,sequence,label,prediction,probability
0,P13123,VKVKFKYKGEEKEVDTSKIKKVWRVGKMVSFTYDDNGKTGRGAVSE...,1,1,0.913370
1,P03211,MSDEGPGTGPGNGLGEKGDTSGPEGSGGSGPQRRGGDNHGRGRGRG...,1,1,0.845794
2,P03023,MKPVTLYDVAEYAGVSYQTVSRVVNQASHVSAKTREKVEAAMAELN...,1,1,0.998892
3,Q38087,MKEFYLTVEQIGDSIFERYIDSNGRERTREVEYKPSLFAHCPESQA...,1,1,0.970120
4,P22670,MATQAYTELQAAPPPSQPPQAPPQAQPQPPPPPPPAAPQPPQPPTA...,1,1,0.999547
...,...,...,...,...,...
2267,P37471,MNFSRERTITEIQNDYKEQVERQNQLKKRRRKGLYRRLTVFGALVF...,0,0,0.017828
2268,P07078,MVVVDKEIKKGQYYLVNGNVVRVTYVNGFDVYYLILKLHKRMICDR...,0,1,0.790979
2269,P16793,MNPSTHVSSNGPTTPPHGPHTTFLPPTSPAPSTSSVAAATLCSPQR...,0,0,0.251041
2270,P80484,MVRSGKKAVVLAAVAFCATSVVQKSHGFVPSPLRQRAAAAGAAAAS...,0,0,0.017818


In [11]:
result = calculate_metrics(scores=df_pred.probability.values,
                           labels=df_pred.label.values,
                           predictions=df_pred.prediction.values,
                           )
pd.Series(result, name=f"DBP-Finder").to_frame().T

,Accuracy,Sensitivity,Specificity,Precision,AUC,F1,MCC
DBP-Finder,0.87588,0.86817,0.883825,0.885057,0.932155,0.876532,0.751918
